In [ ]:
# Joining the flights with the weather data

In [1]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = "/home/jovyan/project"
CLEAN = ROOT + "/data/clean"
OUT = ROOT + "/data/joined"

spark = (SparkSession.builder.appName("dpa-join").config("spark.driver.memory", "2g").config("spark.sql.session.timeZone", "UTC").getOrCreate())

flights = spark.read.parquet(CLEAN + "/flights")
airports = spark.read.parquet(CLEAN + "/airports")
weather = spark.read.parquet(CLEAN + "/weather")
stations = spark.read.parquet(ROOT + "/data/parquet/stations")   # made by ingest.py: iata, icao, station


In [2]:
# saving the number of rows after each step
counts = []
def save_count(step, df):
    n = df.count()
    counts.append((step, n))
    print(step, ":", n)

save_count("clean flights", flights)

clean flights : 13708650


In [ ]:
# airpost data does not provide the timezone so we are usign the timzone of the state

In [3]:
eastern = ["CT", "DE", "DC", "FL", "GA", "IN", "KY", "ME", "MD", "MA", "MI", "NH", "NJ", "NY",
           "NC", "OH", "PA", "RI", "SC", "VT", "VA", "WV"]
central = ["AL", "AR", "IL", "IA", "KS", "LA", "MN", "MS", "MO", "NE", "ND", "OK", "SD", "TN", "TX", "WI"]
mountain = ["CO", "ID", "MT", "NM", "UT", "WY"]
pacific = ["CA", "NV", "OR", "WA"]

origin = airports.select(F.col("iata_code").alias("Origin"),
                         F.col("name").alias("origin_name"),
                         F.regexp_replace("iso_region", "US-", "").alias("state"))



In [4]:
state = F.col("state")
origin = origin.withColumn("timezone", F.when(state.isin(eastern), "America/New_York").when(state.isin(central), "America/Chicago")
                                        .when(state.isin(mountain), "America/Denver")
                                        .when(state.isin(pacific), "America/Los_Angeles")
                                        .when(state == "AZ", "America/Phoenix")
                                        .when(state == "HI", "Pacific/Honolulu")
                                        .when(state == "AK", "America/Anchorage"))

In [5]:
# checking which states did not get a timezone (these airports are removed)

In [6]:
origin.filter(F.col("timezone").isNull()).groupBy("state").count().show()
origin = origin.filter(F.col("timezone").isNotNull())

+-----+-----+
|state|count|
+-----+-----+
+-----+-----+



In [7]:
# building the local departure hour and convret it the UTC

In [8]:
flights = flights.join(origin, "Origin")
save_count("flights with a known origin airport", flights)


flights with a known origin airport : 13564369


In [9]:
flights = flights.withColumn("dep_hour_local", F.to_timestamp(F.concat(F.col("FlightDate"), F.lit(" "), F.lpad("dep_hour", 2, "0")), "yyyy-MM-dd HH"))
flights = flights.withColumn("dep_hour_utc", F.to_utc_timestamp("dep_hour_local", F.col("timezone")))
flights.select("FlightDate", "Origin", "CRSDepTime", "dep_hour_local", "timezone", "dep_hour_utc").show(5, truncate=False)

+----------+------+----------+-------------------+-------------------+-------------------+
|FlightDate|Origin|CRSDepTime|dep_hour_local     |timezone           |dep_hour_utc       |
+----------+------+----------+-------------------+-------------------+-------------------+
|2024-07-07|IAH   |0609      |2024-07-07 06:00:00|America/Chicago    |2024-07-07 11:00:00|
|2024-07-17|JFK   |0600      |2024-07-17 06:00:00|America/New_York   |2024-07-17 10:00:00|
|2024-07-01|LAX   |1255      |2024-07-01 12:00:00|America/Los_Angeles|2024-07-01 19:00:00|
|2024-07-19|JFK   |0600      |2024-07-19 06:00:00|America/New_York   |2024-07-19 10:00:00|
|2024-07-05|LAX   |1255      |2024-07-05 12:00:00|America/Los_Angeles|2024-07-05 19:00:00|
+----------+------+----------+-------------------+-------------------+-------------------+
only showing top 5 rows



In [10]:
# weather station for the departed airport

In [11]:
print("airports with a weather station:", stations.count())
stations.show(50)

airports with a weather station: 49
+----+----+-----------+
|iata|icao|    station|
+----+----+-----------+
| ATL|KATL|72219013874|
| DFW|KDFW|72259003927|
| DEN|KDEN|72565003017|
| ORD|KORD|72530094846|
| CLT|KCLT|72314013881|
| LAX|KLAX|72295023174|
| LAS|KLAS|72386023169|
| PHX|KPHX|72278023183|
| SEA|KSEA|72793024233|
| LGA|KLGA|72503014732|
| MCO|KMCO|72205012815|
| BOS|KBOS|72509014739|
| DCA|KDCA|72405013743|
| SFO|KSFO|72494023234|
| EWR|KEWR|72502014734|
| DTW|KDTW|72537094847|
| JFK|KJFK|74486094789|
| MSP|KMSP|72658014922|
| IAH|KIAH|72243012960|
| SLC|KSLC|72572024127|
| MIA|KMIA|72202012839|
| BNA|KBNA|72327013897|
| BWI|KBWI|72406093721|
| PHL|KPHL|72408013739|
| SAN|KSAN|72290023188|
| FLL|KFLL|74783012849|
| AUS|KAUS|72254013904|
| MDW|KMDW|72534014819|
| TPA|KTPA|72211012842|
| DAL|KDAL|72258013960|
| STL|KSTL|72434013994|
| HNL|PHNL|91182022521|
| RDU|KRDU|72306013722|
| PDX|KPDX|72698024229|
| HOU|KHOU|72244012918|
| SMF|KSMF|72483993225|
| IAD|KIAD|72403093738|
| SJ

In [12]:
flights = flights.join(stations.select(F.col("iata").alias("Origin"), "station"), "Origin")
save_count("flights from the airports with a weather station", flights)

flights from the airports with a weather station : 10762148


In [13]:
# joining the weather of that hour

In [14]:
weather = weather.withColumnRenamed("STATION", "station").withColumnRenamed("hour_utc", "dep_hour_utc")
flights_weather = flights.join(weather, ["station", "dep_hour_utc"])



In [15]:
# saving the result and reading it back, so the checks below do not run the joins again
flights_weather.write.mode("overwrite").partitionBy("Year", "Month").parquet(OUT + "/flights_weather")
flights_weather = spark.read.parquet(OUT + "/flights_weather")
save_count("flights matched with weather", flights_weather)


flights matched with weather : 10734566


In [16]:
flights_weather.select("FlightDate", "Origin", "dep_hour_utc", "ArrDelay", "DepDelay", "visibility", "wind_speed", "rain", "bad_weather").show(5)

+----------+------+-------------------+--------+--------+----------+----------+----+-----------+
|FlightDate|Origin|       dep_hour_utc|ArrDelay|DepDelay|visibility|wind_speed|rain|bad_weather|
+----------+------+-------------------+--------+--------+----------+----------+----+-----------+
|2024-07-13|   BWI|2024-07-13 18:00:00|    -5.0|    -1.0|   16093.0|       1.5| 0.0|      false|
|2024-07-04|   ATL|2024-07-05 01:00:00|   -15.0|    -5.0|   16093.0|       1.5| 0.0|      false|
|2024-07-31|   DEN|2024-07-31 21:00:00|   -13.0|    -7.0|   16093.0|       3.6| 0.0|      false|
|2024-07-18|   ATL|2024-07-19 02:00:00|   147.0|   146.0|   16093.0|       2.6| 0.0|      false|
|2024-07-23|   CLT|2024-07-24 00:00:00|    75.0|    95.0|    8047.0|       3.1| 0.8|      false|
+----------+------+-------------------+--------+--------+----------+----------+----+-----------+
only showing top 5 rows



In [17]:
# checking the joined data

In [18]:
weather_cols = ["wind_speed", "ceiling", "visibility", "temp", "rain", "bad_weather"]
flights_weather.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in weather_cols]).show(vertical=True)

-RECORD 0-------------
 wind_speed  | 12091  
 ceiling     | 2018   
 visibility  | 989    
 temp        | 5689   
 rain        | 308607 
 bad_weather | 0      



In [19]:
# how many flights left in bad and good weather

In [20]:
flights_weather.groupBy("bad_weather").count().show()

+-----------+-------+
|bad_weather|  count|
+-----------+-------+
|       true| 788554|
|      false|9946012|
+-----------+-------+



In [21]:
# summary of the join

In [22]:
summary = pd.DataFrame(counts, columns=["step", "rows_left"])

In [23]:
# rows removed = rows left after the step before - rows left now
summary["rows_removed"] = -summary["rows_left"].diff().fillna(0).astype(int)
print(summary)
os.makedirs(ROOT + "/results", exist_ok=True)
summary.to_csv(ROOT + "/results/join_counts.csv", index=False)

                                               step  rows_left  rows_removed
0                                     clean flights   13708650             0
1               flights with a known origin airport   13564369        144281
2  flights from the airports with a weather station   10762148       2802221
3                      flights matched with weather   10734566         27582
